# 007 · Dunder Methods

> **Syntax › Classes & OOP** · Python 3.14

*Dunder* ("double underscore") methods such as `__repr__`, `__eq__` and `__add__` are the hooks Python calls to make your objects work with built-in syntax: `print`, `==`, `+`, `len()`, `for`, `in`, `[]` and more. They replace a whole zoo of C# concepts — `ToString`, `Equals`/`GetHashCode`, `IComparable`, operator overloads, indexers and `IEnumerable`.

**In this notebook you will learn:**

- `__repr__` / `__str__` vs `ToString()`
- `__eq__` / `__hash__` vs `Equals` / `GetHashCode` — and why defining `__eq__` removes `__hash__`
- ordering with `__lt__` & co. and `functools.total_ordering` vs `IComparable<T>`
- arithmetic operator overloading (`__add__`, `__radd__`, `__mul__`) and `NotImplemented`
- container protocols: `__len__`, `__getitem__`, `__contains__`, `__iter__`
- `__bool__` and `__call__`

## C# mental model

| C# | Python dunder | Triggered by |
|---|---|---|
| `ToString()` | `__repr__`, `__str__` | `repr(x)`, `str(x)`, `print(x)`, f-strings |
| `Equals(object)` / `operator ==` | `__eq__` (and `__ne__`) | `a == b`, `a != b` |
| `GetHashCode()` | `__hash__` | `hash(x)`, dict keys, set members |
| `IComparable<T>.CompareTo` / `operator <` | `__lt__`, `__le__`, `__gt__`, `__ge__` | `<`, `sorted()`, `max()` |
| `operator +`, `operator *` | `__add__`, `__mul__` (+ `__radd__`, `__rmul__`) | `a + b`, `a * 3` |
| `Count` / `Length` | `__len__` | `len(x)` |
| indexer `this[int i]` | `__getitem__`, `__setitem__` | `x[i]`, `x[key]` |
| `Contains(x)` | `__contains__` | `item in x` |
| `IEnumerable<T>.GetEnumerator()` | `__iter__` | `for item in x` |
| `operator true` / implicit bool | `__bool__` | `if x:` |
| delegate / `Invoke` | `__call__` | `x(...)` |

The key mental shift: you never call these methods directly (`len(x)`, not `x.__len__()`). You *implement* them, and Python's syntax and built-in functions call them for you. We'll build up a `Vector` class as we go.

## `__repr__` and `__str__` — string representations

C# has a single `ToString()`. Python has two:

- `__repr__` — **unambiguous, developer-facing**. Ideally looks like the code to recreate the object. Used by the REPL, debugger, logging of containers, and `!r` in f-strings.
- `__str__` — **readable, user-facing**. Used by `print()`, `str()` and f-strings. Falls back to `__repr__` if not defined.

```csharp
// C#
public override string ToString() => $"Vector({X}, {Y})";
```

In [ ]:
class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Vector({self.x!r}, {self.y!r})"

    def __str__(self):
        return f"({self.x}, {self.y})"

v = Vector(1, 2)
print(v)                 # __str__  -> (1, 2)
print(repr(v))           # __repr__ -> Vector(1, 2)
print(f"{v} vs {v!r}")
print([v, Vector(3, 4)]) # containers use __repr__ of their items

**Guideline:** always define `__repr__` (it makes debugging far easier); add `__str__` only when a different user-facing format is useful.

## `__eq__` and `__hash__` — equality and hashing

By default, `==` compares **identity** (like `ReferenceEquals` for a C# class). Define `__eq__` for value equality. Return `NotImplemented` (not `False`) for types you don't know how to compare — Python will then try the other operand's `__eq__` and finally fall back to identity.

`!=` automatically uses the inverse of `__eq__`; you don't need `__ne__`.

```csharp
// C#
public override bool Equals(object? obj) => obj is Vector v && X == v.X && Y == v.Y;
public override int GetHashCode() => HashCode.Combine(X, Y);
```

In [ ]:
class Vector:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __repr__(self):
        return f"Vector({self.x}, {self.y})"

    def __eq__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        return (self.x, self.y) == (other.x, other.y)

print(Vector(1, 2) == Vector(1, 2))   # True — value equality
print(Vector(1, 2) != Vector(3, 4))   # True — derived from __eq__
print(Vector(1, 2) == "something")    # False — NotImplemented -> fallback

⚠️ **Gotcha:** defining `__eq__` **sets `__hash__` to `None`**, making instances unhashable. Python does this deliberately: the default hash is identity-based, which would break the rule *"equal objects must have equal hashes"* (the same contract C# warns about when you override `Equals` without `GetHashCode` — except Python enforces it).

In [ ]:
print(Vector.__hash__)                # None

try:
    {Vector(1, 2)}
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

To make the class hashable, define `__hash__` by hashing a tuple of the same fields used in `__eq__` (the equivalent of `HashCode.Combine`). Only do this for objects that won't change while they're used as keys — ideally **immutable** objects.

In [ ]:
class Vector:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __repr__(self):
        return f"Vector({self.x}, {self.y})"

    def __eq__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        return (self.x, self.y) == (other.x, other.y)

    def __hash__(self):
        return hash((self.x, self.y))      # C#: HashCode.Combine(X, Y)

points = {Vector(1, 2), Vector(1, 2), Vector(3, 4)}
print(points)                              # duplicates removed
print({Vector(0, 0): "origin"}[Vector(0, 0)])

Writing `__init__`, `__repr__`, `__eq__` and `__hash__` by hand is tedious — `@dataclass` generates them for you *(see Classes & OOP › 008 · Dataclasses)*.

## Ordering — `__lt__` and `functools.total_ordering`

The rich comparison methods are `__lt__` (`<`), `__le__` (`<=`), `__gt__` (`>`) and `__ge__` (`>=`). `sorted()`, `min()` and `max()` only need `__lt__`, but for complete comparisons you would need all four. `@functools.total_ordering` fills in the rest from `__eq__` plus **one** of them — a bit like implementing `IComparable<T>.CompareTo` once.

We'll switch to a `Money` example, where ordering is natural:

```csharp
// C#
public readonly record struct Money(decimal Amount, string Currency) : IComparable<Money>
{
    public int CompareTo(Money other) => Amount.CompareTo(other.Amount);
    public static bool operator <(Money a, Money b) => a.CompareTo(b) < 0;
    public static bool operator >(Money a, Money b) => a.CompareTo(b) > 0;
    // ... and <=, >= as well
}
```

In [ ]:
from functools import total_ordering
from decimal import Decimal

@total_ordering
class Money:
    def __init__(self, amount, currency="EUR"):
        self.amount = Decimal(amount)
        self.currency = currency

    def __repr__(self):
        return f"Money('{self.amount}', '{self.currency}')"

    def __eq__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return (self.amount, self.currency) == (other.amount, other.currency)

    def __hash__(self):
        return hash((self.amount, self.currency))

    def __lt__(self, other):
        if not isinstance(other, Money) or other.currency != self.currency:
            return NotImplemented
        return self.amount < other.amount

prices = [Money("9.99"), Money("2.50"), Money("5.00")]
print(sorted(prices))
print(max(prices))
print(Money(3) >= Money(2), Money(3) <= Money(2))   # __ge__/__le__ generated

Returning `NotImplemented` for incompatible values turns a nonsensical comparison into a clear `TypeError`:

In [ ]:
try:
    Money(1, "EUR") < Money(1, "USD")
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

## Arithmetic operators — `__add__`, `__mul__` and the reflected `__radd__`

Each operator maps to a method: `+` → `__add__`, `-` → `__sub__`, `*` → `__mul__`, `/` → `__truediv__`, `//` → `__floordiv__`, `%` → `__mod__`, `**` → `__pow__`, `@` → `__matmul__`, unary `-` → `__neg__`, `abs()` → `__abs__`.

```csharp
// C#
public static Vector operator +(Vector a, Vector b) => new(a.X + b.X, a.Y + b.Y);
public static Vector operator *(Vector v, double k) => new(v.X * k, v.Y * k);
public static Vector operator *(double k, Vector v) => v * k;   // second overload
```

In C# an operator is a static method that can be overloaded for either operand order. In Python, `a + b` first tries `a.__add__(b)`; if that returns `NotImplemented`, it tries the **reflected** method `b.__radd__(a)`. That's how `3 * v` can work even though `int` knows nothing about `Vector`.

In [ ]:
import math

class Vector:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __repr__(self):
        return f"Vector({self.x}, {self.y})"

    def __add__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        return Vector(self.x + other.x, self.y + other.y)

    def __mul__(self, k):
        if not isinstance(k, (int, float)):
            return NotImplemented
        return Vector(self.x * k, self.y * k)

    __rmul__ = __mul__            # k * v — scalar multiplication is commutative

    def __neg__(self):
        return Vector(-self.x, -self.y)

    def __abs__(self):
        return math.hypot(self.x, self.y)

    def __bool__(self):
        return bool(self.x or self.y)    # zero vector is falsy

a, b = Vector(1, 2), Vector(3, 4)
print(a + b, a * 2, 3 * a, -a, abs(b))

`__radd__` is also what makes `sum()` work: `sum()` starts from `0`, so the first step is `0 + Vector(...)`. Supporting it means handling the `0` start value:

In [ ]:
try:
    sum([a, b])
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

print(sum([a, b], Vector(0, 0)))   # option 1: give sum() a start value

class SummableVector(Vector):
    def __radd__(self, other):     # option 2: handle 0 + v
        if other == 0:
            return self
        return NotImplemented

print(sum([SummableVector(1, 2), SummableVector(3, 4)]))

Augmented assignment (`+=`) uses `__iadd__` if defined (for in-place mutation, as `list` does); otherwise it falls back to `__add__` and rebinds the name — which is what you want for immutable value types.

In [ ]:
v = Vector(1, 1)
original = v
v += Vector(1, 0)          # no __iadd__ -> v = v + Vector(1, 0): a NEW object
print(v, original, v is original)

## `__bool__` — truthiness

`if x:` calls `__bool__`; if it's missing, Python uses `__len__` (zero length is falsy); if both are missing, objects are always truthy. Our `Vector` defined `__bool__` above:

In [ ]:
print(bool(Vector(0, 0)), bool(Vector(0, 5)))
if not Vector(0, 0):
    print("zero vector is falsy")

## Container protocols — `__len__`, `__getitem__`, `__contains__`, `__iter__`

Implementing these makes your class behave like a built-in collection, replacing C#'s `Count`, indexers, `Contains` and `IEnumerable<T>`:

```csharp
// C#
public class Playlist : IEnumerable<string>
{
    private readonly List<string> _songs = [];
    public int Count => _songs.Count;
    public string this[int index] => _songs[index];
    public bool Contains(string s) => _songs.Contains(s);
    public IEnumerator<string> GetEnumerator() => _songs.GetEnumerator();
    IEnumerator IEnumerable.GetEnumerator() => GetEnumerator();
}
```

In [ ]:
class Playlist:
    def __init__(self, *songs):
        self._songs = list(songs)

    def __len__(self):                 # len(p)            — C#: Count
        return len(self._songs)

    def __getitem__(self, index):      # p[i], p[1:3]      — C#: indexer
        return self._songs[index]

    def __contains__(self, song):      # "x" in p          — C#: Contains
        return song in self._songs

    def __iter__(self):                # for s in p        — C#: GetEnumerator
        return iter(self._songs)

p = Playlist("intro", "verse", "chorus", "outro")
print(len(p), p[0], p[-1], p[1:3])
print("chorus" in p)
for song in p:
    print(" -", song)

Because `__getitem__` forwards to a list, slicing and negative indexes come for free. `__setitem__` and `__delitem__` support `p[i] = x` and `del p[i]`. `__getitem__` can take any key type — use it for dictionary-like classes, too.

`__iter__` is often written as a generator with `yield` *(see Iterators & Generators › 002 · Generators)*. Note that if you define only `__getitem__`, Python can still iterate and test `in` using index `0, 1, 2, …` — a legacy fallback.

## `__call__` — callable objects

Defining `__call__` lets you call an instance like a function. It's the Python equivalent of an object that holds state and exposes an `Invoke` method (or a closure assigned to a C# delegate).

In [ ]:
class Multiplier:
    def __init__(self, factor):
        self.factor = factor
        self.calls = 0

    def __call__(self, value):
        self.calls += 1
        return value * self.factor

triple = Multiplier(3)
print(triple(5), triple(10))           # call it like a function
print(list(map(triple, [1, 2, 3])))    # usable anywhere a function is expected
print(callable(triple), triple.calls)

## Other dunders worth knowing

| Dunder | Purpose | See |
|---|---|---|
| `__init__`, `__new__` | initialisation / construction | *Classes & OOP › 001* |
| `__enter__`, `__exit__` | `with` statement (C#: `IDisposable` + `using`) | *Error Handling › 003 · Context Managers* |
| `__next__` | iterator protocol | *Iterators & Generators › 001* |
| `__getattr__`, `__setattr__` | intercept attribute access | — |
| `__format__` | custom format specs in f-strings | — |
| `__aenter__`, `__aiter__`, `__await__` | async protocols | *Async › 001* |

## Summary

| Task | C# | Python |
|---|---|---|
| Debug/display text | `ToString()` | `__repr__` (+ `__str__`) |
| Value equality | `Equals`, `operator ==` | `__eq__` (`!=` is derived) |
| Hash code | `GetHashCode()` | `__hash__` (removed when you define `__eq__`) |
| Ordering | `IComparable<T>`, `operator <` | `__lt__` + `@total_ordering` |
| Operators | `static operator +(a, b)` | `__add__` / `__radd__`, `__mul__` / `__rmul__` |
| "I can't handle this type" | overload resolution | `return NotImplemented` |
| Count | `Count` / `Length` | `__len__` |
| Indexer | `this[int i]` | `__getitem__` / `__setitem__` |
| Contains | `Contains(x)` | `__contains__` |
| Enumerate | `IEnumerable<T>` | `__iter__` |
| Truthiness | `operator true` (rare) | `__bool__` |
| Callable object | delegate / `Invoke` | `__call__` |

## Exercises

1. Give `Money` (from above) an `__add__` that adds two amounts of the same currency and returns `NotImplemented` otherwise, plus a `__radd__` so that `sum([Money(1), Money(2)])` works.
2. Write a `Matrix2x2` class whose `__getitem__` accepts a tuple, so that `m[0, 1]` returns the element in row 0, column 1. Give it a useful `__repr__`.
3. Write a `Sentence` class wrapping a string, where `len(s)` is the number of words, `"word" in s` checks for a word, and iterating yields the words.
4. Why does `class P: def __eq__(self, o): return True` make `{P()}` fail? How do you fix it?

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
from decimal import Decimal

# 1
class Money:
    def __init__(self, amount, currency="EUR"):
        self.amount = Decimal(amount)
        self.currency = currency

    def __repr__(self):
        return f"Money('{self.amount}', '{self.currency}')"

    def __add__(self, other):
        if not isinstance(other, Money) or other.currency != self.currency:
            return NotImplemented
        return Money(self.amount + other.amount, self.currency)

    def __radd__(self, other):
        if other == 0:              # sum() starts with 0
            return self
        return NotImplemented

print(sum([Money(1), Money("2.5")]))   # Money('3.5', 'EUR')

# 2
class Matrix2x2:
    def __init__(self, a, b, c, d):
        self._rows = [[a, b], [c, d]]

    def __getitem__(self, pos):
        row, col = pos
        return self._rows[row][col]

    def __repr__(self):
        return f"Matrix2x2({self._rows})"

m = Matrix2x2(1, 2, 3, 4)
print(m[0, 1])                      # 2

# 3
class Sentence:
    def __init__(self, text):
        self._words = text.split()

    def __len__(self):
        return len(self._words)

    def __contains__(self, word):
        return word in self._words

    def __iter__(self):
        return iter(self._words)

s = Sentence("the quick brown fox")
print(len(s), "fox" in s, list(s))

# 4 — defining __eq__ sets __hash__ to None, so instances are unhashable.
#     Fix: also define __hash__ consistently with __eq__ (equal objects must
#     have equal hashes), e.g. `def __hash__(self): return 0` here.
```

</details>